# Ingest File Batch

**Author:** Databricks Ingestion Team  
**Version:** 1.0  
**Last Updated:** 2026-03-31

This notebook ingests files in batch mode. **Default: Incremental Load** (only new data is ingested). Set the load type widget to 'full' for a full reload.

---

**Parameters:**
- `input_path`: Path to the source files
- `output_table`: Destination table name
- `load_type`: 'incremental' (default) or 'full'

---

**Instructions:**
1. Set the input path, output table, and load type using the widgets below.
2. Run all cells to ingest data as per the selected load type.


In [ ]:
# Databricks widgets for parameterization
import pyspark.sql.functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ingest_file_batch")

dbutils.widgets.text("input_path", "", "Input Path")
dbutils.widgets.text("output_table", "bronze_table", "Output Table")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"], "Load Type")
input_path = dbutils.widgets.get("input_path")
output_table = dbutils.widgets.get("output_table")
load_type = dbutils.widgets.get("load_type")

try:
    df = spark.read.format("json").load(input_path)
    logger.info(f"Loaded {df.count()} records from {input_path}")
    if load_type == "full":
        df.write.format("delta").mode("overwrite").saveAsTable(output_table)
        logger.info(f"Full load: Overwrote {output_table}")
    else:
        df.write.format("delta").mode("append").saveAsTable(output_table)
        logger.info(f"Incremental load: Appended to {output_table}")
    print(f"Ingestion ({load_type}) successful.")
except Exception as e:
    logger.error(f"Error in ingestion: {e}")
    print(f"Error: {e}")


## Batch File Ingestion Adapter
Adapter for Bronze RAW ingestion of heterogeneous files.

In [ ]:
_FORMAT_MAP = {
    'json': 'json',
    'jsonl': 'json',
    'binaryFile': 'binaryFile',
    'json_jsonl_mixed': 'binaryFile',
    'csv': 'csv',
    'parquet': 'parquet',
    'avro': 'avro',
    'orc': 'orc',
    'text': 'text',
}

def _reader_format(source: dict, global_config: dict) -> str:
    source_format = str(source.get('source_format', '')).strip()
    file_defaults = global_config.get('connections', {}).get('file_defaults', {})
    default_format = str(file_defaults.get('format', 'binaryFile')).strip()
    return _FORMAT_MAP.get(source_format, default_format)

## Batch File Ingestion Function
Function to read files as a Spark batch DataFrame.

In [ ]:
def ingest_file_batch(spark, source: dict, global_config: dict, source_options: dict | None = None):
    # ...existing code...

---

## Validation & Testing

Below, we validate the notebook logic with a simple assertion to ensure data was ingested. Add more tests as needed for your data.


In [ ]:
# Simple validation
try:
    df_test = spark.table(output_table)
    assert df_test.count() > 0, "No records found in output table."
    print("Validation passed: Data ingested.")
except Exception as e:
    print(f"Validation failed: {e}")


---

## Resource Cleanup

Unpersist DataFrames and clean up resources if needed to avoid memory issues in production workflows.

In [ ]:
# Resource cleanup
if 'df_test' in locals():
    df_test.unpersist(blocking=True)
    print("Resources cleaned up.")
